In [2]:
import pandas as pd
import os
import gzip
import json
import zlib
import numpy as np
import sys


import ast


import numpy as np
import importlib

sys.path.append('data/data_utils')
import data_utils
importlib.reload(data_utils)
from data_utils import *

In [11]:

run_id = "520-rlm-comparison-argparse"
# file_path= f'/checkpoint/maui_sft/winnieyangwn/amaia_dumps/{run_id}/trajectories/mle_bench_bashmle_bench_checkpoint_maui_sft_shared_kniu_datasets_mlebench_full_jsonl/mle_bench_bashmle_bench_checkpoint_maui_sft_shared_kniu_datasets_mlebench_full_jsonl.jsonl'
# file_path = f'/checkpoint/maui_sft/winnieyangwn/amaia_dumps/{run_id}/trajectories/mle_bench_bashmle_bench_checkpoint_maui_sft_shared_kniu_datasets_mlebench_full_jsonl/mle_bench_bashmle_bench_checkpoint_maui_sft_shared_kniu_datasets_mlebench_full_jsonl.jsonl'
file_path = f'/checkpoint/agentic-models/winnieyangwn/amaia_dumps/{run_id}/trajectories/mle_bench_bashmle_bench_checkpoint_maui_sft_winnieyangwn_datasets_gpt5_comparison_xray_jsonl/mle_bench_bashmle_bench_checkpoint_maui_sft_winnieyangwn_datasets_gpt5_comparison_xray_jsonl.jsonl'


In [12]:
df_trj = pd.read_json(file_path, lines=True)


In [13]:
df_trj.iloc[0]["rollouts"][0]["metrics"]["rollout/duration"]

444.3353145830333

In [8]:
df_trj.iloc[0]["rollouts"][0]["traj"]["transitions"][-1]["info"]["pred_solution"]

'import os\nimport sys\nimport time\nimport random\nimport math\nimport csv\nimport json\nimport shutil\nimport subprocess\nfrom pathlib import Path\n\n# Avoid argparse per instructions; use constants/env/dev flag\nDATA_DIR = \'/root/data\'\nTRAIN_CSV = os.path.join(DATA_DIR, \'train.csv\')\nTRAIN_DIR = os.path.join(DATA_DIR, \'train\')\nTEST_DIR = os.path.join(DATA_DIR, \'test\')\nSAMPLE_SUB = os.path.join(DATA_DIR, \'sample_submission.csv\')\nOUT_PATH = \'/workspace/submission.csv\'\nCACHE_DIR = \'/workspace/cache_vinbig\'\nos.makedirs(CACHE_DIR, exist_ok=True)\n\n# Dev flag: create /workspace/dev_run to enable tiny run\nDEV_MODE = os.path.exists(\'/workspace/dev_run\') or os.environ.get(\'DEV_RUN\', \'0\') == \'1\'\nUSE_CLAHE = False  # optional, can be toggled if desired\n\nSEED = 42\nrandom.seed(SEED)\nos.environ[\'PYTHONHASHSEED\'] = str(SEED)\n\n\ndef log(msg):\n    print(f"[vinbig] {msg}", flush=True)\n\n\ndef pip_install_if_needed():\n    """Install required packages if missin

In [9]:

# Flatten the dataframe
df_flat = flatten_dataframe(df_trj)

print(f"Shape: {df_flat.shape}")
print(f"Columns: {df_flat.columns.tolist()}")
df_flat.head()

Shape: (64, 9)
Columns: ['task_name', 'task_description', 'code', 'percentile', 'valid_submission', 'eval_error_output', 'eval_duration', 'rollout_duration', 'rollout']


,task_name,task_description,code,percentile,valid_submission,eval_error_output,eval_duration,rollout_duration,rollout
0,vinbigdata-chest-xray-abnormalities-detection,# Overview\n\n## Description\n\nWhen you have ...,import os\nimport sys\nimport time\nimport ran...,0.0,False,Validation error: Submission invalid! The atte...,0.0,444.335315,"[{'turn_id': 0, 'action': '', 'observation': '..."
1,vinbigdata-chest-xray-abnormalities-detection,# Overview\n\n## Description\n\nWhen you have ...,import os\nimport sys\nimport time\nimport mat...,0.0,False,Validation error: Submission invalid! The atte...,0.0,459.064529,"[{'turn_id': 0, 'action': '', 'observation': '..."
2,vinbigdata-chest-xray-abnormalities-detection,# Overview\n\n## Description\n\nWhen you have ...,import os\nimport sys\nimport time\nimport mat...,0.0,False,Validation error: Submission invalid! The atte...,0.0,492.794865,"[{'turn_id': 0, 'action': '', 'observation': '..."
3,vinbigdata-chest-xray-abnormalities-detection,# Overview\n\n## Description\n\nWhen you have ...,import os\nimport sys\nimport math\nimport ran...,0.0,False,Validation error: Submission invalid! The atte...,0.0,600.401539,"[{'turn_id': 0, 'action': '', 'observation': '..."
4,vinbigdata-chest-xray-abnormalities-detection,# Overview\n\n## Description\n\nWhen you have ...,import os\nimport sys\nimport csv\nimport math...,0.0,False,Validation error: Submission invalid! The atte...,0.0,694.281590,"[{'turn_id': 0, 'action': '', 'observation': '..."


In [ ]:
save_path = f'/checkpoint/agentic-models/winnieyangwn/amaia_dumps/{run_id}/trajectories/{run_id}_metadata.jsonl'

# Save df_flat to jsonl
df_flat.to_json(save_path, orient='records', lines=True)
print(f"Saved {len(df_flat)} rows to {save_path}")

OSError: Cannot save file into a non-existent directory: '/checkpoint/maui_sft/winnieyangwn/amaia_dumps/520-rlm-comparison-argparse/trajectories'

# Data Structure Description

The context is a list of dictionaries, where each dictionary represents one MLE Bench rollout (an agent's attempt to solve a machine learning task). There are 4,800 rollouts in total.

Each rollout dictionary has the following fields:

1. **"task_name"** (str): The unique identifier for the ML task (e.g., "detecting-insults-in-social-commentary")

2. **"task_description"** (str): Full markdown description of the ML task including overview, evaluation criteria, and submission format

3. **"code"** (str | None): The final Python code solution submitted by the agent

4. **"percentile"** (float | None): Performance percentile (0-100) achieved by the submission. Higher is better. 100 means top performer.

5. **"valid_submission"** (bool | None): Whether the agent produced a valid submission file

6. **"eval_error_message"** (str | None): Evaluation result message - contains success info or error details

7. **"eval_duration"** (float | None): GPU execution time in seconds for evaluation

8. **"rollout_duration"** (float | None): Total time in seconds for the entire rollout

9. **"rollout"** (list[dict]): The full conversation trajectory as a list of turns. Each turn has:
   - "turn_id" (int): Turn number starting from 0
   - "action" (str): The agent's action/response (typically bash commands in XML tags)
   - "observation" (str): The environment's response/prompt to the agent

## Example access patterns:
```python
# Get task name
context[0]["task_name"]

# Get percentile
context[0]["percentile"]

# Get number of turns
len(context[0]["rollout"])

# Get first action
context[0]["rollout"][0]["action"]

# Filter successful submissions
[r for r in context if r["valid_submission"] == True]

# Filter by percentile
[r for r in context if r["percentile"] and r["percentile"] >= 50]
```